# Get annotation and genome-context data for a subset of genomes for visualizing strict viral regions

In [205]:
! pip install polars --quiet

In [206]:
from pathlib import Path
OUTPUT_TABLES_DIR = Path("./tables/genome_viz")
OUTPUT_TABLES_DIR.mkdir(exist_ok=True, parents=True)

In [207]:
import os
os.environ["POLARS_MAX_THREADS"] = str(50)
import polars as pl
pl.Config.set_fmt_str_lengths(100)

polars.config.Config

## Load pre-split training data

In [208]:
all_data = (
    pl.read_parquet(
        "/storage2/scratch/kosmopoulos/projects/checkAMG/training_data/processing/genomad_progenomes_combined/training_data_pre_split.parquet"
    )
    .drop([
        "LGBM_viral_prob", "Viral_Origin_Confidence", # Old features used from a model fit on an older model in CheckAMG annotate, not the latest results
    ])
    .select([
        "Protein", "Contig",
        "region_label", "original_region_label", "region_contig_type",
        "True Positive", "True Negative",
        "Source", "Dataset"
    ])
)

In [209]:
all_data

Protein,Contig,region_label,original_region_label,region_contig_type,True Positive,True Negative,Source,Dataset
str,str,str,str,str,bool,bool,str,str
"""1015567_contig_2210_74_1015567_contig_2210|fragment_1_1""","""1015567_contig_2210_74_1015567_contig_2210|fragment_1""","""Viral""","""Viral""","""standalone_virus""",true,false,"""Virus""","""geNomad"""
"""1015567_contig_2210_74_1015567_contig_2210|fragment_1_2""","""1015567_contig_2210_74_1015567_contig_2210|fragment_1""","""Viral""","""Viral""","""standalone_virus""",true,false,"""Virus""","""geNomad"""
"""1015567_contig_2210_74_1015567_contig_2210|fragment_1_3""","""1015567_contig_2210_74_1015567_contig_2210|fragment_1""","""Viral""","""Viral""","""standalone_virus""",true,false,"""Virus""","""geNomad"""
"""1015567_contig_2210_74_1015567_contig_2210|fragment_1_4""","""1015567_contig_2210_74_1015567_contig_2210|fragment_1""","""Viral""","""Viral""","""standalone_virus""",true,false,"""Virus""","""geNomad"""
"""1015567_contig_2210_74_1015567_contig_2210|fragment_1_5""","""1015567_contig_2210_74_1015567_contig_2210|fragment_1""","""Viral""","""Viral""","""standalone_virus""",true,false,"""Virus""","""geNomad"""
…,…,…,…,…,…,…,…,…
"""999547.SAMN02440716.KI421503~1-174096~chromosome_mixed_172""","""999547.SAMN02440716.KI421503~1-174096~chromosome_mixed""","""Nonviral""","""Nonviral""","""chromosome_mixed""",false,true,"""MGE""","""progenomes"""
"""999547.SAMN02440716.KI421503~1-174096~chromosome_mixed_173""","""999547.SAMN02440716.KI421503~1-174096~chromosome_mixed""","""Nonviral""","""Nonviral""","""chromosome_mixed""",false,true,"""MGE""","""progenomes"""
"""999547.SAMN02440716.KI421503~1-174096~chromosome_mixed_174""","""999547.SAMN02440716.KI421503~1-174096~chromosome_mixed""","""Nonviral""","""Nonviral""","""chromosome_mixed""",false,true,"""MGE""","""progenomes"""


## Load genome context and annotations from CheckAMG results

Only load proteins encoded on contigs that have at least one protein that matches to every "step" in the strict/ambiguous viral region algorithm.

In [210]:
CHECKAMG_OUTPUT_DIR = Path("/storage2/scratch/kosmopoulos/projects/checkAMG/training_data/processing/checkamg_annotate_outputs_1.1")
CHECKAMG_GENOMAD_DIR = CHECKAMG_OUTPUT_DIR.joinpath("checkamg_annotate_genomad_dataset")
CHECKAMG_PROGENOMES_DIR = CHECKAMG_OUTPUT_DIR.joinpath("checkamg_annotate_progenomes_dataset")

In [211]:
checkamg_context_schema = pl.read_parquet_schema(CHECKAMG_GENOMAD_DIR.joinpath("results/genes_genomic_context.parquet"))
checkamg_context_schema

{'genome': String,
 'contig': String,
 'protein': String,
 'gene_number': Int64,
 'contig_pos_start': Int64,
 'contig_pos_end': Int64,
 'frame': Int64,
 'is_bin': String,
 'dbCAN_score': Float64,
 'Pfam_score': Float64,
 'METABOLIC_score': Float64,
 'KEGG_score': Float64,
 'CAMPER_score': Float64,
 'PHROG_score': Float64,
 'FOAM_score': Float64,
 'dbCAN_alignment_type': String,
 'Pfam_alignment_type': String,
 'METABOLIC_alignment_type': String,
 'KEGG_alignment_type': String,
 'CAMPER_alignment_type': String,
 'PHROG_alignment_type': String,
 'FOAM_alignment_type': String,
 'dbCAN_hmm_id': String,
 'Pfam_hmm_id': String,
 'METABOLIC_hmm_id': String,
 'KEGG_hmm_id': String,
 'CAMPER_hmm_id': String,
 'PHROG_hmm_id': String,
 'FOAM_hmm_id': String,
 'dbCAN_evalue': Float64,
 'Pfam_evalue': Float64,
 'METABOLIC_evalue': Float64,
 'KEGG_evalue': Float64,
 'CAMPER_evalue': Float64,
 'PHROG_evalue': Float64,
 'FOAM_evalue': Float64,
 'dbCAN_coverage_sequence': Float64,
 'Pfam_coverage_seque

In [212]:
required_cols_context = [
    "protein", "contig",
    "gene_number", "contig_pos_start", "contig_pos_end", "frame",
    "KEGG_V-score", "Pfam_V-score", "PHROG_V-score",
    "window_avg_KEGG_VL-score", "window_avg_Pfam_VL-score", "window_avg_PHROG_VL-score",
] + [col for col in checkamg_context_schema.keys() if col.startswith("step")] + ["viral_region_id"]
required_cols_context

['protein',
 'contig',
 'gene_number',
 'contig_pos_start',
 'contig_pos_end',
 'frame',
 'KEGG_V-score',
 'Pfam_V-score',
 'PHROG_V-score',
 'window_avg_KEGG_VL-score',
 'window_avg_Pfam_VL-score',
 'window_avg_PHROG_VL-score',
 'step1_seed_strict_wvl',
 'step1_in_candidate_region',
 'step2_passes_vscore_gate',
 'step2_in_refined_core',
 'step3_bridge_ok',
 'step3_in_walkback_extended',
 'step4_in_snapped_region',
 'step5_in_merged_region',
 'viral_region_id']

In [216]:
checkamg_context = (
    pl.concat([
        pl.scan_parquet(
            CHECKAMG_GENOMAD_DIR.joinpath("results/genes_genomic_context.parquet")
        )
        .select(required_cols_context),
        pl.scan_parquet(
            CHECKAMG_PROGENOMES_DIR.joinpath("results/genes_genomic_context.parquet"),
        )
        .select(required_cols_context),
    ])
    .collect()
)

In [217]:
checkamg_context

protein,contig,gene_number,contig_pos_start,contig_pos_end,frame,KEGG_V-score,Pfam_V-score,PHROG_V-score,window_avg_KEGG_VL-score,window_avg_Pfam_VL-score,window_avg_PHROG_VL-score,step1_seed_strict_wvl,step1_in_candidate_region,step2_passes_vscore_gate,step2_in_refined_core,step3_bridge_ok,step3_in_walkback_extended,step4_in_snapped_region,step5_in_merged_region,viral_region_id
str,str,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool,bool,bool,bool,i32
"""1015567_contig_2210_74_1015567_contig_2210|fragment_1_1""","""1015567_contig_2210_74_1015567_contig_2210|fragment_1""",1,36,197,1,null,null,null,3.563338,3.90865,3.166663,true,true,false,false,true,false,false,false,null
"""1015567_contig_2210_74_1015567_contig_2210|fragment_1_2""","""1015567_contig_2210_74_1015567_contig_2210|fragment_1""",2,194,412,-1,null,10.0,null,3.563338,3.90865,3.166663,true,true,true,true,true,true,true,false,null
"""1015567_contig_2210_74_1015567_contig_2210|fragment_1_3""","""1015567_contig_2210_74_1015567_contig_2210|fragment_1""",3,520,2157,1,10.0,10.0,10.0,3.563338,3.90865,3.166663,true,true,true,true,true,true,true,false,null
"""1015567_contig_2210_74_1015567_contig_2210|fragment_1_4""","""1015567_contig_2210_74_1015567_contig_2210|fragment_1""",4,2154,2429,1,10.0,10.0,10.0,3.563338,3.90865,3.166663,true,true,true,true,true,true,true,false,null
"""1015567_contig_2210_74_1015567_contig_2210|fragment_1_5""","""1015567_contig_2210_74_1015567_contig_2210|fragment_1""",5,2426,2707,1,null,null,null,3.563338,3.90865,3.166663,true,true,false,false,true,false,false,false,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""999547.SAMN02440716.KI421503~1-174096~chromosome_mixed_172""","""999547.SAMN02440716.KI421503~1-174096~chromosome_mixed""",172,169132,170241,1,10.0,10.0,10.0,3.301602,2.842687,2.785525,true,true,true,true,true,true,true,true,2
"""999547.SAMN02440716.KI421503~1-174096~chromosome_mixed_173""","""999547.SAMN02440716.KI421503~1-174096~chromosome_mixed""",173,170254,171534,-1,0.54,0.44,0.45,3.074284,2.512574,2.839986,true,true,false,true,true,true,true,true,2
"""999547.SAMN02440716.KI421503~1-174096~chromosome_mixed_174""","""999547.SAMN02440716.KI421503~1-174096~chromosome_mixed""",174,171695,172690,-1,10.0,1.33,10.0,3.074284,2.601452,2.839986,true,true,false,true,true,true,true,true,2


In [219]:
checkamg_annot_schema = pl.read_parquet_schema(CHECKAMG_GENOMAD_DIR.joinpath("results/gene_annotations.parquet"))
checkamg_annot_schema

{'Protein': String,
 'Contig': String,
 'Genome': String,
 'Function': String,
 'KEGG_hmm_id': String,
 'FOAM_hmm_id': String,
 'Pfam_hmm_id': String,
 'dbCAN_hmm_id': String,
 'METABOLIC_hmm_id': String,
 'CAMPER_hmm_id': String,
 'PHROG_hmm_id': String,
 'KEGG_score': Float64,
 'FOAM_score': Float64,
 'Pfam_score': Float64,
 'dbCAN_score': Float64,
 'METABOLIC_score': Float64,
 'CAMPER_score': Float64,
 'PHROG_score': Float64,
 'KEGG_coverage': Float64,
 'FOAM_coverage': Float64,
 'Pfam_coverage': Float64,
 'dbCAN_coverage': Float64,
 'METABOLIC_coverage': Float64,
 'CAMPER_coverage': Float64,
 'PHROG_coverage': Float64,
 'KEGG_V-score': Float64,
 'Pfam_V-score': Float64,
 'PHROG_V-score': Float64,
 'Viral_Flanking_Genes_Left_Dist': Float32,
 'Viral_Flanking_Genes_Right_Dist': Float32,
 'MGE_Flanking_Genes_Left_Dist': Float32,
 'MGE_Flanking_Genes_Right_Dist': Float32,
 'step5_in_merged_region': Boolean,
 'nonviral_region_pos': String,
 'Viral_Origin_Confidence': String,
 'KEGG_Descr

In [220]:
ignore_annots = ["METABOLIC", "CAMPER", "dbCAN", "top_hit"] # Ignore some annotations to save space
required_cols_annots = [
    "Protein", "Contig", "Function",
] + [col for col in checkamg_annot_schema.keys() if (col.endswith("_hmm_id") or col.lower().endswith("_description")) and not any(annot in col for annot in ignore_annots)]
required_cols_annots

['Protein',
 'Contig',
 'Function',
 'KEGG_hmm_id',
 'FOAM_hmm_id',
 'Pfam_hmm_id',
 'PHROG_hmm_id',
 'KEGG_Description',
 'FOAM_Description',
 'Pfam_Description',
 'PHROG_Description']

In [221]:
valid_proteins = (
    checkamg_context
    .select(pl.col("protein").alias("Protein"))
    .unique()
).lazy()

checkamg_annots = (
    pl.concat([
        pl.scan_parquet(
            CHECKAMG_GENOMAD_DIR.joinpath("results/gene_annotations.parquet")
        )
        .select(required_cols_annots),
        pl.scan_parquet(
            CHECKAMG_PROGENOMES_DIR.joinpath("results/gene_annotations.parquet"),
        )
        .select(required_cols_annots),
    ])
    .join(valid_proteins, on="Protein", how="inner")
    .collect()
)

In [222]:
checkamg_annots

Protein,Contig,Function,KEGG_hmm_id,FOAM_hmm_id,Pfam_hmm_id,PHROG_hmm_id,KEGG_Description,FOAM_Description,Pfam_Description,PHROG_Description
str,str,str,str,str,str,str,str,str,str,str
"""New_nucc_id_383348|fragment_1_2""","""New_nucc_id_383348|fragment_1""","""Viral RNA-dependent RNA polymerase""",null,null,"""PF00680.26""",null,null,null,"""Viral RNA-dependent RNA polymerase""",null
"""255451.SAMN03891676.LHVJ01000015~1-643952~chromosome_mixed_313""","""255451.SAMN03891676.LHVJ01000015~1-643952~chromosome_mixed""",null,null,null,null,null,null,null,null,null
"""1765722.SAMN04304558.LOCL01000058~1-205869~chromosome_mixed_80""","""1765722.SAMN04304558.LOCL01000058~1-205869~chromosome_mixed""","""pcaH; protocatechuate 3,4-dioxygenase, beta subunit [EC:1.13.11.3]""","""K00449""","""HMMsoil11006""","""PF00775.28""",null,"""pcaH; protocatechuate 3,4-dioxygenase, beta subunit [EC:1.13.11.3]""","""pcaG; protocatechuate 3,4-dioxygenase, alpha subunit [EC:1.13.11.3]; pcaH; protocatechuate 3,4-dioxy…","""Dioxygenase""",null
"""269798.SAMN02598536.CP000383~1-4433218~chromosome_mixed_576""","""269798.SAMN02598536.CP000383~1-4433218~chromosome_mixed""","""asbA; spermidine-citrate ligase [EC:6.3.2.-]""","""K24108""",null,null,null,"""asbA; spermidine-citrate ligase [EC:6.3.2.-]""",null,null,null
"""PLXQ01000071.1|fragment_2_3""","""PLXQ01000071.1|fragment_2""","""serB-plsC; putative phosphoserine phosphatase / 1-acylglycerol-3-phosphate O-acyltransferase [EC:3.1…","""K15781""","""HMMsoil25628""","""PF12710.14""",null,"""serB-plsC; putative phosphoserine phosphatase / 1-acylglycerol-3-phosphate O-acyltransferase [EC:3.1…","""serB, PSPH; phosphoserine phosphatase [EC:3.1.3.3]""","""haloacid dehalogenase-like hydrolase""",null
…,…,…,…,…,…,…,…,…,…,…
"""MICJ01000018.1|fragment_1_3""","""MICJ01000018.1|fragment_1""",null,null,null,null,null,null,null,null,null
"""fragment_17879_11""","""fragment_17879""",null,null,null,null,null,null,null,null,null
"""1761744.SAMN04487912.FNNR01000002~1-639715~chromosome_mixed_467""","""1761744.SAMN04487912.FNNR01000002~1-639715~chromosome_mixed""","""K07018; uncharacterized protein""","""K07018""",null,null,null,"""K07018; uncharacterized protein""",null,null,null


## Combine into one feature table and one sequence table for plotting

Random subsample contigs by 15%, but keep all contigs that had relabelled proteins (those are rarer) to reduce filesize, don't need them all anyway. Mark contigs that had proteins from progenomes re-labeled from Viral to Nonviral using the strict algorithm.

In [277]:
feature_table = (
    checkamg_context
    .join(
        checkamg_annots,
        left_on=["protein", "contig"],
        right_on=["Protein", "Contig"],
        how="inner"
    )
    .join(
        all_data,
        left_on=["protein", "contig"],
        right_on=["Protein", "Contig"],
        how="inner"
    )
    .sort(["Dataset", "contig", "contig_pos_start", "contig_pos_end"])
    .with_columns([
        pl.selectors.integer()
        .exclude(["frame", "gene_number"])
        .cast(pl.Int32),
        pl.col("frame").cast(pl.Int16),
        pl.col("gene_number").cast(pl.Int16),
        pl.selectors.float().cast(pl.Float32),
    ])
)

In [278]:
import random
random.seed(20260501)

genomad_contigs = (
    feature_table
    .filter(pl.col("Dataset") == "geNomad")
    .get_column("contig")
    .unique()
    .sort()
    .to_list()
)
progenomes_contigs = (
    feature_table
    .filter(pl.col("Dataset") == "progenomes")
    .get_column("contig")
    .unique()
    .sort()
    .to_list()
)

relabeled_contigs = (
    feature_table
    .filter(pl.col("region_label")!=pl.col("original_region_label"))
    .get_column("contig")
    .unique()
    .sort()
    .to_list()
)

sampled_contigs = sorted(list(set(
    random.sample(genomad_contigs, round(len(genomad_contigs) * 0.15)) +
    random.sample(progenomes_contigs, round(len(progenomes_contigs) * 0.15)) +
    relabeled_contigs
)))

In [279]:
sampled_contigs[:5], sampled_contigs[-5:], len(sampled_contigs)

(['1000562.SAMN03114893.JSAP01000059~167-104902~standalone_virus',
  '1000565.SAMN02471991.AFHG01000029~267036-303673~standalone_virus',
  '1000565.SAMN02471991.AFHG01000030~143724-146792~standalone_mge',
  '1000565.SAMN02471991.AFHG01000043~80-2316~standalone_mge',
  '1000565.SAMN02471991.AFHG01000044~40759-134788~standalone_virus'],
 ['fragment_9987',
  'fragment_9989',
  'fragment_999',
  'fragment_9992',
  'fragment_9999'],
 226631)

In [280]:
feature_table = feature_table.filter(pl.col("contig").is_in(sampled_contigs))
feature_table

protein,contig,gene_number,contig_pos_start,contig_pos_end,frame,KEGG_V-score,Pfam_V-score,PHROG_V-score,window_avg_KEGG_VL-score,window_avg_Pfam_VL-score,window_avg_PHROG_VL-score,step1_seed_strict_wvl,step1_in_candidate_region,step2_passes_vscore_gate,step2_in_refined_core,step3_bridge_ok,step3_in_walkback_extended,step4_in_snapped_region,step5_in_merged_region,viral_region_id,Function,KEGG_hmm_id,FOAM_hmm_id,Pfam_hmm_id,PHROG_hmm_id,KEGG_Description,FOAM_Description,Pfam_Description,PHROG_Description,region_label,original_region_label,region_contig_type,True Positive,True Negative,Source,Dataset
str,str,i16,i32,i32,i16,f32,f32,f32,f32,f32,f32,bool,bool,bool,bool,bool,bool,bool,bool,i32,str,str,str,str,str,str,str,str,str,str,str,str,bool,bool,str,str
"""1039479_contig_5232_3_1039479_contig_5232|fragment_1_1""","""1039479_contig_5232_3_1039479_contig_5232|fragment_1""",1,1,642,1,null,null,null,NaN,NaN,NaN,false,false,false,false,false,false,false,false,null,null,null,null,null,null,null,null,null,null,"""Viral""","""Viral""","""standalone_virus""",true,false,"""Virus""","""geNomad"""
"""1039479_contig_5232_3_1039479_contig_5232|fragment_1_2""","""1039479_contig_5232_3_1039479_contig_5232|fragment_1""",2,606,902,1,null,null,null,NaN,NaN,NaN,false,false,false,false,false,false,false,false,null,null,null,null,null,null,null,null,null,null,"""Viral""","""Viral""","""standalone_virus""",true,false,"""Virus""","""geNomad"""
"""1039479_contig_5232_3_1039479_contig_5232|fragment_1_3""","""1039479_contig_5232_3_1039479_contig_5232|fragment_1""",3,899,1567,1,null,null,null,NaN,NaN,NaN,false,false,false,false,false,false,false,false,null,null,null,null,null,null,null,null,null,null,"""Viral""","""Viral""","""standalone_virus""",true,false,"""Virus""","""geNomad"""
"""1039479_contig_5232_3_1039479_contig_5232|fragment_1_4""","""1039479_contig_5232_3_1039479_contig_5232|fragment_1""",4,1583,1945,1,null,null,null,NaN,NaN,NaN,false,false,false,false,false,false,false,false,null,null,null,null,null,null,null,null,null,null,"""Viral""","""Viral""","""standalone_virus""",true,false,"""Virus""","""geNomad"""
"""1039479_contig_5232_3_1039479_contig_5232|fragment_1_5""","""1039479_contig_5232_3_1039479_contig_5232|fragment_1""",5,1977,2228,1,null,null,null,NaN,NaN,NaN,false,false,false,false,false,false,false,false,null,null,null,null,null,null,null,null,null,null,"""Viral""","""Viral""","""standalone_virus""",true,false,"""Virus""","""geNomad"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""999547.SAMN02440716.KI421500~1-3984464~chromosome_mixed_3735""","""999547.SAMN02440716.KI421500~1-3984464~chromosome_mixed""",3735,3979276,3980427,-1,10.0,10.0,10.0,2.930434,2.368829,3.089206,true,true,true,true,true,true,true,true,21,"""ParA-like partition protein""","""K03496""",null,"""PF13614.13""","""phrog_164""","""parA, soj; chromosome partitioning protein""",null,"""AAA domain""","""ParA-like partition protein""","""Nonviral""","""Nonviral""","""chromosome_mixed""",false,true,"""Host""","""progenomes"""
"""999547.SAMN02440716.KI421500~1-3984464~chromosome_mixed_3736""","""999547.SAMN02440716.KI421500~1-3984464~chromosome_mixed""",3736,3980710,3980871,-1,null,null,null,2.930434,2.45592,3.089206,true,true,false,true,true,true,true,true,21,null,null,null,null,null,null,null,null,null,"""Nonviral""","""Nonviral""","""chromosome_mixed""",false,true,"""Host""","""progenomes"""
"""999547.SAMN02440716.KI421500~1-3984464~chromosome_mixed_3737""","""999547.SAMN02440716.KI421500~1-3984464~chromosome_mixed""",3737,3981091,3982326,1,null,10.0,null,2.930434,2.45592,3.089206,true,true,true,true,true,true,true,true,21,null,null,null,null,null,null,null,null,null,"""Nonviral""","""Nonviral""","""chromosome_mixed""",false,true,"""Host""","""progenomes"""


In [281]:
feature_table.write_parquet(OUTPUT_TABLES_DIR.joinpath("genes_feature_table.parquet"))

In [285]:
sequence_table = (
    feature_table
    .with_columns([
        (pl.col("region_label") != pl.col("original_region_label")).alias("_relabeled"),
        pl.col("viral_region_id").is_not_null().alias("_in_strict_viral_region"),
    ])
    .group_by(["contig", "Dataset", "region_contig_type"])
    .agg([
        pl.col("contig_pos_end").max().cast(pl.Int32).alias("length"),
        pl.col("viral_region_id").n_unique().cast(pl.Int32).alias("n_viral_regions"),
        pl.len().cast(pl.Int32).alias("n_ptns_total"),

        (pl.col("Source") == "Virus").sum().cast(pl.Int32).alias("n_viral_ptns"),
        (pl.col("Source") == "Host").sum().cast(pl.Int32).alias("n_host_ptns"),
        (pl.col("Source") == "MGE").sum().cast(pl.Int32).alias("n_mge_ptns"),

        (pl.col("True Positive") == True).sum().cast(pl.Int32).alias("n_true_viral"),

        pl.col("_relabeled").sum().cast(pl.Int32).alias("n_ptns_relabeled"),
        pl.col("_in_strict_viral_region").sum().cast(pl.Int32).alias("n_ptns_strict_viral_region"),
    ])
    .with_columns([
        (pl.col("n_viral_ptns") / pl.col("n_ptns_total")).cast(pl.Float32).alias("pct_source_viral_ptns"),
        (pl.col("n_host_ptns") / pl.col("n_ptns_total")).cast(pl.Float32).alias("pct_source_host_ptns"),
        (pl.col("n_mge_ptns") / pl.col("n_ptns_total")).cast(pl.Float32).alias("pct_source_mge_ptns"),
        (pl.col("n_true_viral") / pl.col("n_ptns_total")).cast(pl.Float32).alias("pct_true_viral"),
        (pl.col("n_ptns_relabeled") / pl.col("n_ptns_total")).cast(pl.Float32).alias("pct_ptns_relabeled"),
        (pl.col("n_ptns_strict_viral_region") / pl.col("n_ptns_total")).cast(pl.Float32).alias("pct_ptns_strict_viral_region"),
    ])
    .rename({"Dataset": "dataset"})
    .sort(["dataset", "contig"])
)

In [286]:
sequence_table

contig,dataset,region_contig_type,length,n_viral_regions,n_ptns_total,n_viral_ptns,n_host_ptns,n_mge_ptns,n_true_viral,n_ptns_relabeled,n_ptns_strict_viral_region,pct_source_viral_ptns,pct_source_host_ptns,pct_source_mge_ptns,pct_true_viral,pct_ptns_relabeled,pct_ptns_strict_viral_region
str,str,str,i32,i32,i32,i32,i32,i32,i32,i32,i32,f32,f32,f32,f32,f32,f32
"""1039479_contig_5232_3_1039479_contig_5232|fragment_1""","""geNomad""","""standalone_virus""",4199,1,12,12,0,0,12,0,0,1.0,0.0,0.0,1.0,0.0,0.0
"""1085434_contig_2176_7_1085434_contig_2176|fragment_1""","""geNomad""","""standalone_virus""",3906,1,8,8,0,0,8,0,0,1.0,0.0,0.0,1.0,0.0,0.0
"""1085436_contig_405_6_1085436_contig_405|fragment_1""","""geNomad""","""standalone_virus""",6590,2,11,11,0,0,11,0,6,1.0,0.0,0.0,1.0,0.0,0.545455
"""1093096_contig_1717_4_1093096_contig_1717|fragment_1""","""geNomad""","""standalone_virus""",3153,1,6,6,0,0,6,0,0,1.0,0.0,0.0,1.0,0.0,0.0
"""10H_2_19940_cov_32_0|fragment_1""","""geNomad""","""standalone_virus""",5340,1,10,10,0,0,10,0,0,1.0,0.0,0.0,1.0,0.0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""999541.SAMN02603164.CP002599~1-4413616~chromosome_mixed""","""progenomes""","""chromosome_mixed""",4413285,23,3887,1632,1486,769,278,1354,493,0.419861,0.3823,0.197839,0.07152,0.348341,0.126833
"""999541.SAMN02603164.CP002600~1-3700833~chromosome_mixed""","""progenomes""","""chromosome_mixed""",3700565,10,3096,1007,1149,940,12,995,97,0.325258,0.371124,0.303618,0.003876,0.321382,0.031331
"""999543.SAMN02261279.KB905360~1-59197~chromosome_mixed""","""progenomes""","""chromosome_mixed""",59196,2,69,0,1,68,0,0,52,0.0,0.014493,0.985507,0.0,0.0,0.753623


In [289]:
sequence_table.write_parquet(OUTPUT_TABLES_DIR.joinpath("genes_sequence_table.parquet"))